In [2]:
import pandas as pd 
import prince
import mca
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns 
import statsmodels.api as sm
from sklearn.utils import resample

In [5]:
#SECTION 1: prep data for MCA explaining variation in telework/employment correlation
allcat = pd.read_csv('allcat.csv')
people_r = pd.read_csv('people.csv')
p_mca = people_r[['TUCASEID','TULINENO','CORR']]
p_mca['CORR'].describe()

count    276303.000000
mean          1.410021
std           0.491838
min           1.000000
25%           1.000000
50%           1.000000
75%           2.000000
max           2.000000
Name: CORR, dtype: float64

In [8]:
#resample dataset to 200,000 random samples, half CORR=1 and half CORR=2
cor = p_mca[p_mca['CORR'] == 1]
uncor = p_mca[p_mca['CORR'] == 2]

cor_r = resample(cor, replace=False, n_samples=100000, random_state=1)
uncor_r = resample(uncor, replace=False, n_samples=100000, random_state=1)

p_resampled = pd.concat([cor_r, uncor_r])
p_resampled

,TUCASEID,TULINENO,CORR
261901,20231007231568,3,1
272061,20231209231836,1,1
159609,20220101220694,1,1
24644,20190908190029,2,1
243634,20230604230590,1,1
...,...,...,...
268823,20231109231436,1,2
235984,20230403231291,1,2
222916,20230201230554,2,2
129635,20210605212065,1,2


In [11]:
#add other columns to remaining dataset
mcadf=p_resampled.merge(allcat, on=['TUCASEID','TULINENO'],how='left')
mcadf.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200000 entries, 0 to 199999
Data columns (total 58 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   TUCASEID      200000 non-null  int64  
 1   TULINENO      200000 non-null  int64  
 2   CORR          200000 non-null  int64  
 3   Unnamed: 0.2  200000 non-null  int64  
 4   Unnamed: 0.1  200000 non-null  int64  
 5   Unnamed: 0    200000 non-null  int64  
 6   GEDIV         200000 non-null  int64  
 7   GESTFIPS      200000 non-null  int64  
 8   GTCBSA        200000 non-null  int64  
 9   GTMETSTA      200000 non-null  int64  
 10  HEHOUSUT      200000 non-null  int64  
 11  HETENURE      200000 non-null  int64  
 12  HRHTYPE       200000 non-null  int64  
 13  HRYEAR4       200000 non-null  int64  
 14  HUSPNISH      200000 non-null  int64  
 15  PEABSRSN      200000 non-null  int64  
 16  PEAFEVER      200000 non-null  int64  
 17  PEAFNOW       200000 non-null  int64  
 18  PEAF

In [14]:
#drop columns not relevant to assessing who is most likely to have correlated telework and emoloyment statuses
mcadf = mcadf.drop(['Unnamed: 0.1','Unnamed: 0','GTCBSA','HRYEAR4','PEAFWHN1','PEAFWHN2','PEAFWHN3','PEAFWHN4','PECERT2','PEDWRSN','PEGR6COR','PEHGCOMP','PEMJNUM','PEMLR','PENATVTY','PRDASIAN','PRDTHSP','PRMARSTA','PUAFEVER'], axis=1)
mcadf.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200000 entries, 0 to 199999
Data columns (total 39 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   TUCASEID      200000 non-null  int64  
 1   TULINENO      200000 non-null  int64  
 2   CORR          200000 non-null  int64  
 3   Unnamed: 0.2  200000 non-null  int64  
 4   GEDIV         200000 non-null  int64  
 5   GESTFIPS      200000 non-null  int64  
 6   GTMETSTA      200000 non-null  int64  
 7   HEHOUSUT      200000 non-null  int64  
 8   HETENURE      200000 non-null  int64  
 9   HRHTYPE       200000 non-null  int64  
 10  HUSPNISH      200000 non-null  int64  
 11  PEABSRSN      200000 non-null  int64  
 12  PEAFEVER      200000 non-null  int64  
 13  PEAFNOW       200000 non-null  int64  
 14  PECERT1       200000 non-null  int64  
 15  PECERT3       200000 non-null  int64  
 16  PECYC         200000 non-null  int64  
 17  PEDIPGED      200000 non-null  int64  
 18  PEDI

In [17]:
mcadf.to_csv('thesis_mca_data.csv', header=True)

In [ ]:
#SECTION 2: run MCA

In [20]:
count = pd.read_table('thesis_mca_data.csv', sep=',', skiprows=0, index_col=0, header=0)
count = count.astype('category') 
print(count.shape)

(200000, 39)


In [29]:
#import cleaned dataset and drop na values
data = pd.read_table('thesis_mca_data.csv',
                     sep=',', skiprows=0, index_col=0, header=0)
data.dropna(inplace=True)
data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 49966 entries, 22 to 199998
Data columns (total 39 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   TUCASEID      49966 non-null  int64  
 1   TULINENO      49966 non-null  int64  
 2   CORR          49966 non-null  int64  
 3   Unnamed: 0.2  49966 non-null  int64  
 4   GEDIV         49966 non-null  int64  
 5   GESTFIPS      49966 non-null  int64  
 6   GTMETSTA      49966 non-null  int64  
 7   HEHOUSUT      49966 non-null  int64  
 8   HETENURE      49966 non-null  int64  
 9   HRHTYPE       49966 non-null  int64  
 10  HUSPNISH      49966 non-null  int64  
 11  PEABSRSN      49966 non-null  int64  
 12  PEAFEVER      49966 non-null  int64  
 13  PEAFNOW       49966 non-null  int64  
 14  PECERT1       49966 non-null  int64  
 15  PECERT3       49966 non-null  int64  
 16  PECYC         49966 non-null  int64  
 17  PEDIPGED      49966 non-null  int64  
 18  PEDISDRS      49966 non-null 

In [32]:
#remove id and target variables
X = data.drop(['CORR','TUCASEID','TULINENO','Unnamed: 0.2'], axis=1) 
cols_list = ['GEDIV', 'GESTFIPS', 'GTMETSTA', 'HEHOUSUT', 'HETENURE','HRHTYPE','HUSPNISH','PEABSRSN','PEAFEVER','PEAFNOW','PECERT1','PECERT3','PECYC','PEDIPGED','PEDISDRS','PEDISEAR','PEDISEYE','PEDISOUT','PEDISPHY','PEDISREM','PEERNCOV','PEERNHRY','PEGRPROF','PEHSPNOW','PEMARITL','PEMJOT','PESEX','PRCITSHP','PRDISFLG','PTDTRACE','PUBUS1','HRNUMHOU_c','PRERNHLY_c','PRNMCHLD_C','PRTAGE_C']
X.columns = cols_list
X.shape

(49966, 35)

In [35]:
mca = prince.MCA(
    n_components=3,
    n_iter=3,
    copy=True,
    check_input=True,
    engine='sklearn',
    random_state=42
)
mca = mca.fit(X) 
mca = mca.transform(X) 
print(mca)

               0         1         2
22     -0.252185  0.002019  0.186355
32     -0.041382  0.298247  0.096288
33      0.408711 -0.462776 -0.180358
37      0.032445 -0.313180 -0.106527
51      0.302937 -0.532547 -0.108100
...          ...       ...       ...
199993 -0.134394  0.028059  0.264853
199995 -0.063719 -0.077707  0.079825
199996 -0.594715  0.029257 -0.045213
199997 -0.028348 -0.125501 -0.128388
199998 -0.281759  0.096530  0.005697

[49966 rows x 3 columns]


In [41]:
#ran with different setting to find the number of components with significant eigenvalues
mca = prince.MCA(n_components=50)
mca = mca.fit(X)
eig = mca.eigenvalues_summary
#with 35 columns, any eigenvalue > 0.29 is significant

In [47]:
eig.to_csv('eig.csv', header=True)

In [ ]:
#SECTION 3; MCA results

In [50]:
mca.scree_plot()

/opt/conda/envs/anaconda-2024.02-py310/lib/python3.10/site-packages/altair/utils/core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)


alt.Chart(...)

In [56]:
#mca for top 50 components
results = mca.column_coordinates(X)
results.to_csv('mca-results.csv', header=True)
results

,0,1,2,3,4,5,6,7,8,9,...,40,41,42,43,44,45,46,47,48,49
GEDIV__1,-0.332859,0.084166,-0.029490,-0.978427,-0.184807,-1.491007,-0.455783,-0.538289,1.059034,-0.296889,...,0.030197,0.041114,-0.013568,-0.006899,0.037647,0.010507,-0.019759,-0.006870,-0.005699,0.026853
GEDIV__2,-0.042103,-0.066681,0.223854,-1.188843,0.579510,0.041794,-1.392190,-0.830981,0.879401,0.905578,...,-0.000757,-0.009830,-0.027702,0.007028,0.017458,-0.009297,-0.026175,-0.012799,0.005485,-0.000880
GEDIV__3,-0.337754,0.559762,-0.177937,0.182685,0.606899,-0.173228,-0.561609,0.307572,-1.862930,-0.495734,...,0.004688,0.022232,-0.003775,0.026419,0.028014,-0.012147,-0.002600,-0.017202,0.008310,-0.003296
GEDIV__4,-0.366609,0.672512,-0.411714,1.002868,1.324908,-0.484481,0.209334,1.143091,1.030083,1.228274,...,-0.007333,0.021253,-0.012089,0.031221,0.016262,0.010321,-0.014174,-0.020407,-0.002566,0.018036
GEDIV__5,0.322288,0.067060,-0.554300,-0.506451,-0.773744,0.920568,-0.066231,1.196249,0.446795,-0.596501,...,-0.015982,-0.028424,-0.015986,-0.044177,-0.045965,-0.004751,-0.008109,0.017656,0.007616,-0.021184
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
PRTAGE_C__3,-0.283363,-0.211976,0.041250,0.199908,-0.127458,0.141178,-0.000424,0.072655,0.027838,-0.002557,...,-0.072612,-0.048668,0.056424,-0.031690,-0.135405,-0.185828,0.145796,0.158839,-0.188014,-0.029739
PRTAGE_C__4,-0.422932,-0.158969,0.127490,0.102498,-0.170185,0.269272,-0.053560,0.009755,0.088144,-0.000775,...,-0.021567,0.231497,0.042551,0.121015,-0.005231,0.173898,-0.162637,0.046826,0.125583,0.024006
PRTAGE_C__5,-0.133652,0.232252,0.080882,-0.273240,0.003844,0.021594,0.076046,-0.113922,-0.015203,-0.003979,...,0.324457,-0.222253,-0.169468,-0.194781,0.222124,-0.005767,0.036715,-0.319389,0.127666,-0.149577
PRTAGE_C__6,0.054321,0.597152,0.088434,-0.410313,0.121391,-0.115109,0.129329,-0.151733,0.016959,0.118531,...,-0.387591,0.227315,0.195021,0.103879,-0.257275,-0.003943,-0.095921,0.300327,-0.190121,0.192995


In [97]:
for x in range (50):
    print('component ', x, ' variable coordinates >1')
    #print significant positive contributors
    print(results[x].loc[results[x] > 1])

component  0  variable coordinates >1
GESTFIPS__11    1.301584
HEHOUSUT__4     1.387042
HEHOUSUT__10    1.004765
HRHTYPE__10     1.387773
PEAFEVER__-1    1.161022
PEDISDRS__1     2.704788
PEDISEYE__1     1.561331
PEDISOUT__1     2.488779
PEDISPHY__1     1.498623
PEDISREM__1     1.725525
PRDISFLG__1     1.300654
PTDTRACE__12    2.245027
PTDTRACE__14    1.302883
PRTAGE_C__1     1.041965
Name: 0, dtype: float64
component  1  variable coordinates >1
PEDISDRS__1     5.046569
PEDISEAR__1     2.557346
PEDISEYE__1     2.503618
PEDISOUT__1     3.690004
PEDISPHY__1     2.949193
PEDISREM__1     2.599063
PRDISFLG__1     2.465037
PTDTRACE__14    1.377738
PTDTRACE__23    1.537576
Name: 1, dtype: float64
component  2  variable coordinates >1
GESTFIPS__15    1.152032
HEHOUSUT__4     1.506533
HEHOUSUT__12    1.078676
HRHTYPE__9      1.060090
PEABSRSN__12    1.229864
PEDISDRS__1     7.019173
PEDISEAR__1     2.898900
PEDISEYE__1     3.444364
PEDISOUT__1     5.423583
PEDISPHY__1     3.863628
PEDISREM__1  

In [9]:
#keep only coordinates >1 or <-1
sig_res = results[abs(results) > 1]
sig_res = sig_res.dropna(how='all', axis=1)
sig_res = sig_res.dropna(how='all', axis=0)
a_s_r = sig_res.transpose()
sig_res

NameError: name 'results' is not defined

In [68]:
list(a_s_r.columns)

['GEDIV__1',
 'GEDIV__2',
 'GEDIV__4',
 'GEDIV__5',
 'GEDIV__6',
 'GEDIV__7',
 'GEDIV__8',
 'GESTFIPS__1',
 'GESTFIPS__2',
 'GESTFIPS__4',
 'GESTFIPS__5',
 'GESTFIPS__8',
 'GESTFIPS__9',
 'GESTFIPS__10',
 'GESTFIPS__11',
 'GESTFIPS__12',
 'GESTFIPS__13',
 'GESTFIPS__15',
 'GESTFIPS__16',
 'GESTFIPS__17',
 'GESTFIPS__19',
 'GESTFIPS__20',
 'GESTFIPS__21',
 'GESTFIPS__22',
 'GESTFIPS__23',
 'GESTFIPS__24',
 'GESTFIPS__25',
 'GESTFIPS__26',
 'GESTFIPS__27',
 'GESTFIPS__28',
 'GESTFIPS__29',
 'GESTFIPS__30',
 'GESTFIPS__31',
 'GESTFIPS__32',
 'GESTFIPS__33',
 'GESTFIPS__34',
 'GESTFIPS__35',
 'GESTFIPS__36',
 'GESTFIPS__37',
 'GESTFIPS__38',
 'GESTFIPS__40',
 'GESTFIPS__41',
 'GESTFIPS__42',
 'GESTFIPS__44',
 'GESTFIPS__45',
 'GESTFIPS__46',
 'GESTFIPS__47',
 'GESTFIPS__48',
 'GESTFIPS__49',
 'GESTFIPS__50',
 'GESTFIPS__51',
 'GESTFIPS__53',
 'GESTFIPS__54',
 'GESTFIPS__55',
 'GESTFIPS__56',
 'GTMETSTA__3',
 'HEHOUSUT__2',
 'HEHOUSUT__3',
 'HEHOUSUT__4',
 'HEHOUSUT__5',
 'HEHOUSUT__6',
 'H

In [71]:
var = []
subcat = []

In [74]:
#determine which variables are positively correlated to CORR variable
for x in list(a_s_r.columns):
    y = str(x)
    z = y.split('__')[0]
    a = y.split('__')[1]
    var.append(z)
    subcat.append(a)

In [86]:
s_res['var'] = var
s_res['subcat'] = subcat
s_res

,var,subcat
GEDIV__1,GEDIV,1
GEDIV__2,GEDIV,2
GEDIV__4,GEDIV,4
GEDIV__5,GEDIV,5
GEDIV__6,GEDIV,6
...,...,...
HRNUMHOU_c__6,HRNUMHOU_c,6
PRNMCHLD_C__4,PRNMCHLD_C,4
PRNMCHLD_C__5,PRNMCHLD_C,5
PRTAGE_C__1,PRTAGE_C,1


In [89]:
s_res.loc[s_res['var'] == 'PRNMCHLD_C', 'var'] = 'PRNMCHLD_c'
s_res.loc[s_res['var'] == 'PRTAGE_C', 'var'] = 'PRTAGE_c'
s_res

,var,subcat
GEDIV__1,GEDIV,1
GEDIV__2,GEDIV,2
GEDIV__4,GEDIV,4
GEDIV__5,GEDIV,5
GEDIV__6,GEDIV,6
...,...,...
HRNUMHOU_c__6,HRNUMHOU_c,6
PRNMCHLD_C__4,PRNMCHLD_c,4
PRNMCHLD_C__5,PRNMCHLD_c,5
PRTAGE_C__1,PRTAGE_c,1
